# C2 classifier — DeBERTa-v3-base fine-tune on Colab

Tiny launcher. The actual training code lives in `mostargate/classifier/train.py` in the repo; this notebook just bootstraps a Colab T4 instance and runs that module.

**Before running:**
1. `Runtime → Change runtime type → T4 GPU` (free tier).
2. `Runtime → Run all`.
3. When done, download `classifier_model.zip` from the **Files** panel on the left and unzip into `dataset/classifier_artifacts/model/` locally.

Expected wall time on T4: 3–5 minutes including model download.

## 1. Pull the repo

In [1]:
!git clone https://github.com/0xballistics/mostargate.git
%cd mostargate
!git pull

fatal: destination path 'mostargate' already exists and is not an empty directory.
/content/mostargate
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 9 (delta 8), reused 9 (delta 8), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 1.70 KiB | 290.00 KiB/s, done.
From https://github.com/0xballistics/mostargate
   54f834f..160406a  main       -> origin/main
Updating 54f834f..160406a
Fast-forward
 docs/phase_c_classifier_findings.md |  8 ++++----
 mostargate/classifier/train.py      | 12 +++++++++++-
 notebooks/colab_train.ipynb         | 17 +++++++++++++++--
 3 files changed, 30 insertions(+), 7 deletions(-)


## 2. Install Python deps

Colab images already include `torch` and `numpy`. We add the few packages `mostargate.classifier.train` imports beyond that.

In [2]:
!pip install -q transformers datasets sentencepiece protobuf python-dotenv

## 3. Verify GPU is attached

In [3]:
!nvidia-smi

Sun Jun 21 19:17:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 4. Train

Hyperparameters are baked into `train.py` per `docs/phase_c_classifier_findings.md` §2 — 5 epochs, batch 16 (GPU), lr 2e-5, max_len 256, seed 42, BCEWithLogitsLoss with per-class pos_weight clipped at 10. No internal validation hold-out: all 500 records train the model, save the final checkpoint after epoch 5.

The first run downloads `microsoft/deberta-v3-base` (~700 MB) from the HuggingFace hub. Subsequent re-runs in the same session use the cached download.

In [8]:
!python -m mostargate.classifier.train --epochs 20 --learning-rate 5e-5

device=cuda batch=16 epochs=20 lr=5e-05 seed=42 fp16=False
Loaded 500 training records.
Label matrix: shape=(500, 15) dtype=float32 mean_positives_per_record=2.50
Loading weights: 100% 198/198 [00:00<00:00, 28325.39it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_prediction

## 5. Test model after training

In [7]:
import json, torch, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from mostargate.constants import TOOLS

PERMISSIONS = list(TOOLS.keys())
MODEL_DIR  = "dataset/classifier_artifacts/model"

tok   = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).cuda().eval()

test  = json.load(open("dataset/test.json"))
enc   = tok([r["prompt"] for r in test], padding=True, truncation=True,
            max_length=256, return_tensors="pt").to("cuda")
with torch.no_grad():
    probs = torch.sigmoid(model(**enc).logits).cpu().numpy()   # (100, 15)

# 1. Distribution of probabilities — bimodal means it learned, mush near 0.5 means it didn't
print(f"shape={probs.shape}  mean={probs.mean():.3f}  std={probs.std():.3f}\n")
for lo, hi in [(0,.05),(.05,.20),(.20,.40),(.40,.60),(.60,.80),(.80,.95),(.95,1.01)]:
    n   = ((probs >= lo) & (probs < hi)).sum()
    pct = 100*n/probs.size
    print(f"[{lo:.2f}, {hi:.2f}): {pct:5.1f}%  {'#'*int(pct/2)}")

# 2. Per-permission F1 at threshold 0.5
gt = np.array([[r["permissions"].get(p, False) for p in PERMISSIONS] for r in test], dtype=int)
preds = (probs >= 0.5).astype(int)
print(f"\n{'permission':<22}{'gt+':>5}{'pred+':>7}{'TP':>4}{'FP':>4}{'FN':>4}{'P':>6}{'R':>6}{'F1':>6}")
for j, p in enumerate(PERMISSIONS):
    tp = ((gt[:,j]==1) & (preds[:,j]==1)).sum()
    fp = ((gt[:,j]==0) & (preds[:,j]==1)).sum()
    fn = ((gt[:,j]==1) & (preds[:,j]==0)).sum()
    P  = tp / max(tp+fp, 1); R = tp / max(tp+fn, 1)
    F1 = 2*P*R / max(P+R, 1e-9)
    print(f"{p:<22}{gt[:,j].sum():>5}{preds[:,j].sum():>7}{tp:>4}{fp:>4}{fn:>4}{P:>6.2f}{R:>6.2f}{F1:>6.2f}")

macro_f1 = np.mean([
    (lambda tp,fp,fn: 2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0)(
        ((gt[:,j]==1) & (preds[:,j]==1)).sum(),
        ((gt[:,j]==0) & (preds[:,j]==1)).sum(),
        ((gt[:,j]==1) & (preds[:,j]==0)).sum(),
    )
    for j in range(15)
])
print(f"\nmacro-F1 at threshold 0.5: {macro_f1:.3f}")
print(f"(reference: TF-IDF=0.70, Claude=0.89)")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

shape=(100, 15)  mean=0.462  std=0.085

[0.00, 0.05):   0.0%  
[0.05, 0.20):   6.7%  ###
[0.20, 0.40):   6.7%  ###
[0.40, 0.60):  86.7%  ###########################################
[0.60, 0.80):   0.0%  
[0.80, 0.95):   0.0%  
[0.95, 1.01):   0.0%  

permission              gt+  pred+  TP  FP  FN     P     R    F1
github_read              29    100  29  71   0  0.29  1.00  0.45
pull_request_create       1      0   0   0   1  0.00  0.00  0.00
code_execute              7      0   0   0   7  0.00  0.00  0.00
confluence_read          30      0   0   0  30  0.00  0.00  0.00
jira_read                19      0   0   0  19  0.00  0.00  0.00
jira_write               22    100  22  78   0  0.22  1.00  0.36
slack_read               10    100  10  90   0  0.10  1.00  0.18
slack_write               6      0   0   0   6  0.00  0.00  0.00
salesforce_read          12      0   0   0  12  0.00  0.00  0.00
database_read            44    100  44  56   0  0.44  1.00  0.61
email_read               20    100

**bold text**## 6. Zip the trained model for download

Right-click `classifier_model.zip` in the file panel on the left → `Download`. Unzip locally so the directory ends up at `dataset/classifier_artifacts/model/`.

In [5]:
!zip -qr classifier_model.zip dataset/classifier_artifacts/model/
!ls -lh classifier_model.zip
print("\nDownload classifier_model.zip from the file panel (left sidebar).")

-rw-r--r-- 1 root root 324M Jun 21 19:18 classifier_model.zip

Download classifier_model.zip from the file panel (left sidebar).
